# Executive Function Task Performance, System Usability, and Thematic Annotations in Undergraduate Students Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the *Executive Function Task Performance, System Usability, and Thematic Annotations in Undergraduate Students* FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by the Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.f2na-3dct/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.f2na-3dct/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
List available record sets, fields, and their IDs.

Below we print the available record sets within the dataset, along with their `@id` and the associated fields (and their `@id`s, which you'll need for further data selection).

**Tip:** All references to record sets, fields, and columns should use their `@id` per the Croissant specification.

In [ ]:
# List all record sets and fields (referenced by their @id)

print("Available record sets in the dataset:\n")
for recset in metadata.record_sets:
    print(f"- RecordSet name: {recset.name}")
    print(f"  @id: {recset.id}")
    print(f"  Description: {recset.description}")
    print(f"  Fields:@ids and names:")
    for field in recset.fields:
        print(f"    - {field.id}: {field.name} (Type: {field.data_type})")
    print("")

## 3. Data Extraction
Let's load data from several of the main record sets into DataFrames for further analysis.

Below, choose specific record sets and use their `@id`. Each resulting DataFrame uses the `@id` as its key in the dictionary for easy access.

In [ ]:
# Select relevant record sets by @id (replace these with the actual record set @ids to inspect the data)
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # Load records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with {len(df)} records and columns:")
        print(df.columns.tolist())
        print(df.head(), "\n")
    else:
        print(f"Record set {record_set_id} contains no records or could not be loaded.\n")

## 4. Exploratory Data Analysis (EDA)
Let's run some simple processing and exploration on a numeric field of interest in one of the record sets.

Replace `<example_record_set_id>` and `<example_numeric_field_id>` below with the `@id` of a record set and a numeric field that appeared in the overview above. You may also select example fields that represent demographic or outcome data.

We'll perform filtering, normalization, and grouping.

In [ ]:
# Example record set selection (replace with actual @id suitable for numeric analysis from the cell above)

example_record_set_id = record_sets[0] if record_sets else None # Use the first available record set

if example_record_set_id and not dataframes[example_record_set_id].empty:
    df = dataframes[example_record_set_id]
    print(f"Analyzing record set: {example_record_set_id}\nColumns: {df.columns.tolist()}")

    # Try to pick a numeric field in this record set
    # Here we simply use the first float/integer-like field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    
    if numeric_field is None:
        print("No numeric field found to demonstrate EDA. Please update the field selection above.")
    else:
        print(f"Using numeric field for EDA: {numeric_field}")

        # Example threshold for demonstration
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        # Filter rows where value > mean
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold} (mean):\n", filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field found for grouping analysis.")
else:
    print("No records found for the selected record set.")

## 5. Visualization
Let's plot the distribution of our selected numeric field.

Visualizations depend on your filtering and grouping above. Update the plotting fields as appropriate for the dataset and fields you are interested in.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure previous analysis found a suitable dataframe/field
if example_record_set_id and numeric_field:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {example_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No suitable numeric field available for visualization.")

## 6. Conclusion
We demonstrated how to load, explore, and process the *Executive Function Task Performance, System Usability, and Thematic Annotations* dataset using the `mlcroissant` library, referencing all data by their Croissant `@id` fields.
You can adapt this notebook by selecting different record sets and fields (by `@id`), applying alternative filters, and running additional analyses relevant to your research or exploration goals.

For more on the dataset, see its [FAIR^2 landing page](https://sen.science/doi/10.71728/senscience.f2na-3dct) and the [mlcroissant documentation](https://mlcommons.org/croissant/spec/).
